In [1]:
!pip install kaggle sentence-transformers faiss-cpu pandas google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.8 MB/s eta 0:00:00


In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"shampabarman","key":"8b478a23aba036d9a481153a002dfc21"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets list

ref                                                         title                                                  size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------  -----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dmahajanbe23/bmw-global-automotive-sales                    BMW Global Automotive Sales                           55017  2026-02-22 18:18:38.170000           3394         65  1.0              
shree0910/online-vs-in-store-shopping-behaviour-dataset     Online vs In-Store Shopping Behaviour Dataset        354896  2026-02-18 08:16:20.137000           1803         42  1.0              
krupalpatel07/gold-price-dynamics                           Gold Price Dynamics                                   85982  2026-03-03 05:42:46.960000            626         24  1.0              
likithagedipudi/starbucks-customer-

In [5]:
!kaggle datasets download -d kanchana1990/ev-battery-qc-synthetic-defect-dataset
!unzip ev-battery-qc-synthetic-defect-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/kanchana1990/ev-battery-qc-synthetic-defect-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
  0% 0.00/401k [00:00<?, ?B/s]
100% 401k/401k [00:00<00:00, 901MB/s]
Archive:  ev-battery-qc-synthetic-defect-dataset.zip
  inflating: ev_battery_qc_data_2026_kaggle.csv  


In [6]:
!ls

ev_battery_qc_data_2026_kaggle.csv	    kaggle.json
ev-battery-qc-synthetic-defect-dataset.zip  sample_data


In [7]:
import pandas as pd

df = pd.read_csv("ev_battery_qc_data_2026_kaggle.csv")

print(df.head())
print(df.columns)

       Cell_ID  Batch_ID Production_Line    Shift        Supplier  \
0  CELL-004920  BTH-0001          Line_3  Evening        ChemCorp   
1  CELL-014782  BTH-0001          Line_1    Night        ChemCorp   
2  CELL-019348  BTH-0001          Line_2    Night       LithioMat   
3  CELL-008537  BTH-0001          Line_2  Morning  VoltIndustries   
4  CELL-010539  BTH-0001          Line_1  Evening  VoltIndustries   

   Ambient_Temp_C  Anode_Overhang_mm  Electrolyte_Volume_ml  \
0           20.76              0.108                  14.81   
1           22.38              0.126                  14.96   
2           20.18              0.135                  14.97   
3           24.30              0.162                  14.84   
4           22.22              0.130                  14.89   

   Internal_Resistance_mOhm  Capacity_mAh  Retention_50Cycle_Pct  \
0                     14.01        4980.0                  97.79   
1                     14.70        4989.0                  97.35   
2 

In [8]:
documents = []

for _, row in df.iterrows():
    text = f"""
    Cell ID: {row['Cell_ID']}
    Batch: {row['Batch_ID']}
    Production Line: {row['Production_Line']}
    Shift: {row['Shift']}
    Supplier: {row['Supplier']}
    Ambient Temperature: {row['Ambient_Temp_C']} C
    Anode Overhang: {row['Anode_Overhang_mm']} mm
    Electrolyte Volume: {row['Electrolyte_Volume_ml']} ml
    Internal Resistance: {row['Internal_Resistance_mOhm']} mOhm
    Capacity: {row['Capacity_mAh']} mAh
    Capacity Retention after 50 cycles: {row['Retention_50Cycle_Pct']} %
    Defect Type: {row['Defect_Type']}
    Inspector Comment: {row['Inspector_Comment'] if pd.notna(row['Inspector_Comment']) else "No comment"}
    QC Grade: {row['QC_Grade']}
    """

    documents.append(text)

print("Total documents:", len(documents))
print(documents[0])

Total documents: 20000

    Cell ID: CELL-004920
    Batch: BTH-0001
    Production Line: Line_3
    Shift: Evening
    Supplier: ChemCorp
    Ambient Temperature: 20.76 C
    Anode Overhang: 0.108 mm
    Electrolyte Volume: 14.81 ml
    Internal Resistance: 14.01 mOhm
    Capacity: 4980.0 mAh
    Capacity Retention after 50 cycles: 97.79 %
    Defect Type: Poor Retention
    Inspector Comment: No comment
    QC Grade: Scrap
    


In [9]:
!pip install sentence-transformers faiss-cpu

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (20000, 384)


In [11]:
import faiss
import numpy as np

# convert embeddings to numpy
embeddings = np.array(embeddings).astype("float32")

# create FAISS index
index = faiss.IndexFlatL2(384)

# add embeddings to index
index.add(embeddings)

print("Total vectors in database:", index.ntotal)

Total vectors in database: 20000


In [12]:
def search(query, top_k=3):

    # convert query to embedding
    query_embedding = model.encode([query])

    # search in FAISS
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

In [13]:
query = "Which battery cells have high internal resistance?"

results = search(query)

for r in results:
    print(r)
    print("---------------")


    Cell ID: CELL-000001
    Batch: BTH-0103
    Production Line: Line_1
    Shift: Night
    Supplier: VoltIndustries
    Ambient Temperature: 20.66 C
    Anode Overhang: 0.132 mm
    Electrolyte Volume: 14.82 ml
    Internal Resistance: 15.16 mOhm
    Capacity: 4998.0 mAh
    Capacity Retention after 50 cycles: 96.54 %
    Defect Type: High Internal Resistance
    Inspector Comment: No comment
    QC Grade: Grade A
    
---------------

    Cell ID: CELL-009300
    Batch: BTH-0062
    Production Line: Line_3
    Shift: Night
    Supplier: VoltIndustries
    Ambient Temperature: 22.74 C
    Anode Overhang: 0.173 mm
    Electrolyte Volume: 14.9 ml
    Internal Resistance: 15.12 mOhm
    Capacity: 4937.0 mAh
    Capacity Retention after 50 cycles: 95.09 %
    Defect Type: High Internal Resistance
    Inspector Comment: No comment
    QC Grade: Grade A
    
---------------

    Cell ID: CELL-011102
    Batch: BTH-0026
    Production Line: Line_3
    Shift: Night
    Supplier: VoltIndust

In [14]:
!pip install transformers sentence-transformers faiss-cpu

In [15]:
from transformers import pipeline

qa_model = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [16]:
response = qa_model("Explain battery internal resistance in simple words")

print(response[0]["generated_text"])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain battery internal resistance in simple words


In [17]:
def rag_answer(question, top_k=3):

    # convert question to embedding
    query_embedding = model.encode([question]).astype("float32")

    # search in FAISS index
    distances, indices = index.search(query_embedding, top_k)

    # collect context from retrieved documents
    context = ""
    for idx in indices[0]:
        context += documents[idx] + "\n\n"

    # create prompt for LLM
    prompt = f"""
Use the following battery dataset records to answer the question.

Context:
{context}

Question: {question}

Answer:
"""

    response = qa_model(prompt, max_new_tokens=150)

    return response[0]["generated_text"]

In [18]:
question = "Which battery cells have poor retention?"

answer = rag_answer(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Which battery cells have poor retention?

Answer:

Use the following battery dataset records to answer the question.

Context:

    Cell ID: CELL-017560
    Batch: BTH-0083
    Production Line: Line_3
    Shift: Night
    Supplier: VoltIndustries
    Ambient Temperature: 21.51 C
    Anode Overhang: 0.1 mm
    Electrolyte Volume: 15.05 ml
    Internal Resistance: 14.82 mOhm
    Capacity: 4978.0 mAh
    Capacity Retention after 50 cycles: 95.97 %
    Defect Type: Poor Retention
    Inspector Comment: Routine visual inspection passed.
    QC Grade: Scrap
    


    Cell ID: CELL-016018
    Batch: BTH-0227
    Production Line: Line_3
    Shift: Night
    Supplier: ChemCorp
    Ambient Temperature: nan C
    Anode Overhang: 0.09 mm
    Electrolyte Volume: 15.07 ml
    Internal Resistance: 14.29 mOhm
    Capacity: 5014.0 mAh
    Capacity Retention after 50 cycles: 96.98 %
    Defect Type: Poor Retention
    Inspector Comment: No comment
    QC Grade: Scrap
    


    Cell ID: CELL-

In [19]:
question = "Show scrap battery cells?"

answer = rag_answer(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Show scrap battery cells?

Answer:

Use the following battery dataset records to answer the question.

Context:

    Cell ID: CELL-019906
    Batch: BTH-0049
    Production Line: Line_1
    Shift: Night
    Supplier: VoltIndustries
    Ambient Temperature: 20.9 C
    Anode Overhang: 0.112 mm
    Electrolyte Volume: 15.05 ml
    Internal Resistance: 14.79 mOhm
    Capacity: 5031.0 mAh
    Capacity Retention after 50 cycles: 97.54 %
    Defect Type: nan
    Inspector Comment: No comment
    QC Grade: Scrap
    


    Cell ID: CELL-014280
    Batch: BTH-0489
    Production Line: Line_1
    Shift: Night
    Supplier: VoltIndustries
    Ambient Temperature: 21.1 C
    Anode Overhang: 0.083 mm
    Electrolyte Volume: 14.92 ml
    Internal Resistance: 15.52 mOhm
    Capacity: 4894.0 mAh
    Capacity Retention after 50 cycles: 96.6 %
    Defect Type: High Internal Resistance
    Inspector Comment: No comment
    QC Grade: Scrap
    


    Cell ID: CELL-007557
    Batch: BTH-0458
    

In [20]:
question = "Which supplier produced defective batteries?"

answer = rag_answer(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Which supplier produced defective batteries?

Answer:

Use the following battery dataset records to answer the question.

Context:

    Cell ID: CELL-008092
    Batch: BTH-0429
    Production Line: Line_2
    Shift: Night
    Supplier: LithioMat
    Ambient Temperature: 22.22 C
    Anode Overhang: 0.107 mm
    Electrolyte Volume: 15.16 ml
    Internal Resistance: 13.85 mOhm
    Capacity: 5022.0 mAh
    Capacity Retention after 50 cycles: 95.87 %
    Defect Type: nan
    Inspector Comment: Operator noted irregular machine vibration.
    QC Grade: Scrap
    


    Cell ID: CELL-005314
    Batch: BTH-0434
    Production Line: Line_3
    Shift: Night
    Supplier: LithioMat
    Ambient Temperature: 22.38 C
    Anode Overhang: 0.111 mm
    Electrolyte Volume: 14.89 ml
    Internal Resistance: 14.22 mOhm
    Capacity: 5000.0 mAh
    Capacity Retention after 50 cycles: 96.52 %
    Defect Type: nan
    Inspector Comment: Operator noted irregular machine vibration.
    QC Grade: Scrap

In [21]:
question = "Which supplier produced defective batteries?"

answer = rag_answer(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Which supplier produced defective batteries?

Answer:

Use the following battery dataset records to answer the question.

Context:

    Cell ID: CELL-008092
    Batch: BTH-0429
    Production Line: Line_2
    Shift: Night
    Supplier: LithioMat
    Ambient Temperature: 22.22 C
    Anode Overhang: 0.107 mm
    Electrolyte Volume: 15.16 ml
    Internal Resistance: 13.85 mOhm
    Capacity: 5022.0 mAh
    Capacity Retention after 50 cycles: 95.87 %
    Defect Type: nan
    Inspector Comment: Operator noted irregular machine vibration.
    QC Grade: Scrap
    


    Cell ID: CELL-005314
    Batch: BTH-0434
    Production Line: Line_3
    Shift: Night
    Supplier: LithioMat
    Ambient Temperature: 22.38 C
    Anode Overhang: 0.111 mm
    Electrolyte Volume: 14.89 ml
    Internal Resistance: 14.22 mOhm
    Capacity: 5000.0 mAh
    Capacity Retention after 50 cycles: 96.52 %
    Defect Type: nan
    Inspector Comment: Operator noted irregular machine vibration.
    QC Grade: Scrap